# DoublePageScanSplitter v2

Splits double-page scans into a left (verso) and a right (recto) page.

**New in v2:** the split is defined by **two clicks** on the gutter (one near the top, one near the bottom),
so slanted/skewed gutters are handled.

Input formats: JPEG, PNG and TIFF. The split pages are saved in the format of the source image.

Controls in the image window:
* **Left click** twice: top and bottom point of the gutter (order does not matter). The window closes automatically after the second click.
* **Right click**: do not split this image.
* **r**: reset the clicks on the current image.
* **Esc** (or closing the window): stop collecting; the images clicked so far are still processed.

Split modes (`MODE`):
* `'rotate'`: the image is rotated so the gutter becomes vertical, then split straight. Both pages come out deskewed.
* `'cut'`: no rotation and no resampling. The image is cut along the slanted line and the triangle that belongs to the other page is filled with `FILL`.

In [ ]:
import os
import cv2
import numpy as np
import time
import pandas as pd
from datetime import datetime

In [ ]:
# get the starting time of the process
st = time.time()

In [ ]:
sourceDir = r"..\data\images" # update source directory
targetDir = r"..\data\split" # update target directory

MODE = 'rotate'       # 'rotate' (deskew, then split) or 'cut' (split along the slanted line, no resampling)
FILL = (255, 255, 255) # BGR colour for areas without image content (white)
MIN_ANGLE = 0.1        # degrees; below this the gutter is treated as vertical (no rotation)
JPEG_QUALITY = 95      # only for JPEG output; PNG and TIFF are saved lossless
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.tif', '.tiff') # the split pages keep the format of the source image

In [ ]:
filepaths = []
for subdir, dirs, files in os.walk(sourceDir):
    for file in files:
        if file.lower().endswith(IMAGE_EXTS):
            sourceFilepath = os.path.join(subdir, file)
            filepaths.append(sourceFilepath)
        else:
            print('not a jpg, png or tif:', file)
print(len(filepaths))

In [ ]:
# imread/imwrite via numpy so that paths with umlauts etc. work on Windows
# images are read as 8 bit colour (grey and 16 bit images are converted; of a multi-page TIFF only the first page is read)
def read_image(path):
    img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError('cannot read image: ' + path)
    return img

def write_image(path, img):
    # the format follows the file extension
    ext = os.path.splitext(path)[1].lower()
    params = {'.jpg': [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY], '.jpeg': [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY],
              '.png': [cv2.IMWRITE_PNG_COMPRESSION, 6],
              '.tif': [cv2.IMWRITE_TIFF_COMPRESSION, 8], '.tiff': [cv2.IMWRITE_TIFF_COMPRESSION, 8]}[ext] # 8 = Deflate
    ok, buf = cv2.imencode(ext, img, params)
    buf.tofile(path)

In [ ]:
def order_points(p1, p2):
    # returns (top point, bottom point)
    return tuple(sorted([p1, p2], key=lambda p: p[1]))

def gutter_angle(p1, p2):
    # angle of the gutter against the vertical, in degrees (positive = bottom leans to the right)
    (x1, y1), (x2, y2) = order_points(p1, p2)
    return float(np.degrees(np.arctan2(x2 - x1, y2 - y1)))

def line_x_at(p1, p2, y):
    # x-coordinate of the (extended) gutter line at height y
    (x1, y1), (x2, y2) = order_points(p1, p2)
    if y2 == y1:
        return (x1 + x2) / 2
    return x1 + (y - y1) * (x2 - x1) / (y2 - y1)

def split_vertical(img, x):
    x = int(round(x))
    return img[:, :x], img[:, x:]

def split_rotate(img, p1, p2):
    h, w = img.shape[:2]
    (x1, y1), (x2, y2) = order_points(p1, p2)
    angle = gutter_angle(p1, p2)
    mx, my = (x1 + x2) / 2, (y1 + y2) / 2
    if abs(angle) < MIN_ANGLE:
        return split_vertical(img, mx)
    # rotate around the midpoint of the gutter so that the gutter becomes vertical
    M = cv2.getRotationMatrix2D((mx, my), -angle, 1.0)
    # enlarge the canvas so no image content is cut off
    corners = M @ np.array([[0, w, 0, w], [0, 0, h, h], [1, 1, 1, 1]], dtype=np.float64)
    minx, miny = corners.min(axis=1)
    maxx, maxy = corners.max(axis=1)
    M[0, 2] -= minx
    M[1, 2] -= miny
    nw, nh = int(np.ceil(maxx - minx)), int(np.ceil(maxy - miny))
    rot = cv2.warpAffine(img, M, (nw, nh), flags=cv2.INTER_CUBIC,
                         borderMode=cv2.BORDER_CONSTANT, borderValue=FILL)
    # the rotation centre stays fixed, so after the shift the gutter is at x = mx - minx
    return split_vertical(rot, mx - minx)

def split_cut(img, p1, p2):
    h, w = img.shape[:2]
    x_top, x_bot = line_x_at(p1, p2, 0), line_x_at(p1, p2, h)
    if abs(x_bot - x_top) < 1:
        return split_vertical(img, (x_top + x_bot) / 2)
    line = line_x_at(p1, p2, np.arange(h) + 0.5)[:, None]  # gutter x per row
    cols = np.arange(w) + 0.5
    left_w = int(np.clip(np.ceil(max(x_top, x_bot)), 1, w))
    right_x = int(np.clip(np.floor(min(x_top, x_bot)), 0, w - 1))
    left = img[:, :left_w].copy()
    left[cols[None, :left_w] >= line] = FILL
    right = img[:, right_x:].copy()
    right[cols[None, right_x:] < line] = FILL
    return left, right

def split_image(img, p1, p2, mode=MODE):
    return split_rotate(img, p1, p2) if mode == 'rotate' else split_cut(img, p1, p2)

In [ ]:
WIN = "DoublePageScanSplitter"

def collect_points(filepath):
    # returns [(x, y), (x, y)], None (= do not split) or 'ABORT'
    img = read_image(filepath)
    h, w = img.shape[:2]
    lw = max(2, w // 600)  # line width relative to the image size
    state = {'pts': [], 'skip': False, 'disp': img.copy()}

    def redraw():
        disp = img.copy()
        for (x, y) in state['pts']:
            cv2.circle(disp, (x, y), lw * 4, (255, 0, 255), -1)
        if len(state['pts']) == 2:
            p1, p2 = state['pts']
            top = (int(round(line_x_at(p1, p2, 0))), 0)
            bot = (int(round(line_x_at(p1, p2, h))), h)
            cv2.line(disp, top, bot, (255, 0, 255), lw)
            cv2.putText(disp, '%.2f deg' % gutter_angle(p1, p2), (state['pts'][0][0] + 15, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (255, 0, 255), lw)
        state['disp'] = disp

    def on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN and len(state['pts']) < 2:
            state['pts'].append((x, y))
            print(x, y)
            redraw()
        elif event == cv2.EVENT_RBUTTONDOWN:
            print('None')
            state['skip'] = True

    cv2.namedWindow(WIN, cv2.WINDOW_NORMAL)
    scale = min(1200 / w, 800 / h)
    cv2.resizeWindow(WIN, int(w * scale), int(h * scale))
    cv2.setWindowTitle(WIN, os.path.basename(filepath) + '  -  2 clicks on the gutter | right click: no split | r: reset | Esc: stop')
    cv2.setMouseCallback(WIN, on_mouse)
    while True:
        cv2.imshow(WIN, state['disp'])
        key = cv2.waitKey(20) & 0xFF
        if state['skip']:
            return None
        if len(state['pts']) == 2:
            cv2.imshow(WIN, state['disp'])
            cv2.waitKey(400)  # show the line briefly
            return list(state['pts'])
        if key == ord('r'):
            state['pts'] = []
            redraw()
        if key == 27 or cv2.getWindowProperty(WIN, cv2.WND_PROP_VISIBLE) < 1:
            return 'ABORT'

In [ ]:
points = [] # per image: [(x1, y1), (x2, y2)] or None (= no split)

for filepath in filepaths:
    result = collect_points(filepath)
    if result == 'ABORT':
        print('stopped by user after', len(points), 'images')
        break
    points.append(result)
cv2.destroyAllWindows()

In [ ]:
records = []
for filepath, pts in zip(filepaths, points):
    print(filepath) # shows progress; can be commented out
    if pts is None:
        records.append({'Filepaths': filepath, 'x1': None, 'y1': None, 'x2': None, 'y2': None, 'angle': None, 'mode': None})
        continue
    destFilepath = os.path.join(targetDir, os.path.relpath(filepath, sourceDir))
    os.makedirs(os.path.dirname(destFilepath), exist_ok=True)
    root, ext = os.path.splitext(destFilepath)
    img = read_image(filepath)
    img_verso, img_recto = split_image(img, *pts)
    write_image(root + '_L_verso' + ext, img_verso) # can be commented out if necessary
    write_image(root + '_R_recto' + ext, img_recto) # can be commented out if necessary
    (x1, y1), (x2, y2) = order_points(*pts)
    records.append({'Filepaths': filepath, 'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2,
                    'angle': round(gutter_angle(*pts), 3), 'mode': MODE})

img.shape returns (Height, Width, Number of Channels) <br>
Slicing a Numpy array: cropped = img[start_row:end_row, start_col:end_col] <br>
For a vertical split at x this means img[:, :x] for the left page and img[:, x:] for the right page.
In `'rotate'` mode the image is first rotated around the midpoint of the two clicked points by the gutter angle,
so the gutter becomes vertical; in `'cut'` mode each pixel is assigned to the side of the slanted line it lies on.

In [ ]:
# get the end time
et = time.time()

# get the execution time
elapsed_time = et - st
strftime = time.strftime("%H:%M:%S", time.gmtime(elapsed_time))
print('Execution time:', elapsed_time, 'seconds')
print('Execution time:', strftime)

In [ ]:
# count number of files in targetDir (incl. subfolders)
targetDirFileCount = sum(len([f for f in files if f.lower().endswith(IMAGE_EXTS)]) for _, _, files in os.walk(targetDir))
print(targetDirFileCount)

dt_string = datetime.now().strftime("%d/%m/%Y %H:%M:%S")
executor = os.getlogin()

In [ ]:
timing_dict = {'sourceDir': sourceDir, 'sourceDirFileCount': len(filepaths), 'targetDir': targetDir, 'targetDirFileCount': targetDirFileCount, 'elapsedTime': elapsed_time, 'strftime': strftime, 'dt_string': dt_string, 'executor': executor}
print(timing_dict)
df = pd.DataFrame.from_dict([timing_dict])
df.to_csv(os.path.join(targetDir, 'timing.csv'), mode='a', index=False, header=False)

In [ ]:
# two points per image -> new file splitValues_v2.csv (header is written on first use)
df2 = pd.DataFrame(records, columns=['Filepaths', 'x1', 'y1', 'x2', 'y2', 'angle', 'mode'])
splitValuesPath = os.path.join(targetDir, 'splitValues_v2.csv')
df2.to_csv(splitValuesPath, mode='a', index=False, header=not os.path.exists(splitValuesPath))
df2